# Create an Agent with external API call enabled

In this exercise, you'll build an agent that can interact with external APIs to gather real-time data
and provide responses based on that information. You'll combine concepts from state management and
memory while adding the ability to make external API calls safely and effectively.


## Challenge

Your task is to create an agent that can make External API Calls:

- Implement tools that interact with real APIs
- Handle API responses and errors gracefully
- Use environment variables for API keys
- Process and format API data for user consumption

## Setup
First, let's import the necessary libraries:

In [1]:
import os
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(), override=True)
assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY is not set"
assert os.getenv("OPENAI_BASE_URL"), "OPENAI_BASE_URL is not set"

In [2]:
import os
import random
from typing import List
import requests
from dotenv import load_dotenv

from lib.agents import Agent
from lib.messages import BaseMessage
from lib.tooling import tool

In [3]:
load_dotenv()

True

## Define API tools

Feel free to use any open service available through APIs.

Here are a few examples. You can follow the instructions given.
- https://jsonplaceholder.typicode.com/guide/
- https://www.exchangerate-api.com/
- https://openweathermap.org/

Or you can find one you're interested in here.
- https://github.com/public-apis/public-apis

In [4]:
@tool
def get_random_pokemon() -> dict:
    """Get a random Pokemon from the original 151"""
    URL = "https://pokeapi.co/api/v2/pokemon?limit=151"
    response = requests.get(URL)
    response.raise_for_status()
    return random.choice(response.json()['results'])

In [12]:
@tool
def get_exchange_rate(from_currency: str = "USD") -> dict:
    """
    Get latest exchange rates from a base currency
    args:
        from_currency (str): Base currency code (default: USD)
    """
    API_KEY = os.getenv("EXCHANGERATE_API_KEY")
    BASE_URL = "https://v6.exchangerate-api.com/v6"
    
    url = f"{BASE_URL}/{API_KEY}/latest/{from_currency}"
    response = requests.get(url)
    response.raise_for_status()
    return response.json()

In [13]:
@tool
def get_weather(city: str) -> dict:
    """
    Get current weather for a city using OpenWeather API
    args:
        city (str): Name of the city to get weather for
    """
    API_KEY = os.getenv("OPENWEATHER_API_KEY")
    BASE_URL = "https://api.openweathermap.org/data/2.5/weather"
    
    params = {
        "q": city,
        "appid": API_KEY,
        "units": "metric"
    }
    
    response = requests.get(BASE_URL, params=params)
    response.raise_for_status()
    return response.json()

In [14]:
tools = [get_weather, get_exchange_rate, get_random_pokemon]

In [15]:
agent = Agent(
    model_name="gpt-4o-mini",
    instructions=(
        "You are an assistant that can help with:\n"
        "1. Getting weather information for cities\n"
        "2. Checking currency exchange rates\n"
        "3. Getting a random Pokemon\n"
        "Use the available tools to help answer questions about these topics.\n"
        "Maintain context across conversations within the same session."
    ),
    tools=tools
)

In [16]:
def print_messages(messages: List[BaseMessage]):
    for m in messages:
        print(f" -> (role = {m.role}, content = {m.content}, tool_calls = {getattr(m, 'tool_calls', None)})")

## Run your Agent

In [8]:
# Change the query and then run your agent
query = "Pick one random Pokemon!"
session_id = "external_tools"

In [9]:
run1 = agent.invoke(
    query=query, 
    session_id=session_id,
)

print("\nMessages from run 1:")
messages = run1.get_final_state()["messages"]
print_messages(messages)

[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__

Messages from run 1:
 -> (role = system, content = You are an assistant that can help with:
, tool_calls = None)
 -> (role = user, content = Pick one random Pokemon!, tool_calls = None)
 -> (role = assistant, content = None, tool_calls = [ChatCompletionMessageFunctionToolCall(id='call_5hGjDCIWdFgqsMQ7zvaTrN3P', function=Function(arguments='{}', name='get_random_pokemon'), type='function')])
 -> (role = tool, content = "{'name': 'gastly', 'url': 'https://pokeapi.co/api/v2/pokemon/92/'}", tool_calls = None)
 -> (role = assistant, content = I picked a random Pokémon for you: **Gastly**! 

You can find more about Gastly [here](https://pokeapi.co/api/v2/pokemon/92/)., tool_calls = None)


In [19]:
run3 =  agent.invoke(
    query="What's the weather like in London?", 
    session_id=session_id,
)

print("\nMessages from run 3:")
messages = run1.get_final_state()["messages"]
print_messages(messages)

[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor


KeyboardInterrupt: 

## Check session histories

In [10]:
runs = agent.get_session_runs(session_id)
for i, run_object in enumerate(runs, 1):
    print(f"\n# Run {i}", run_object.metadata)
    print("Messages:")
    print_messages(run_object.get_final_state()["messages"])


# Run 1 {'run_id': '48794dc0-deed-4104-9334-e1077cbb9a3a', 'start_timestamp': '2026-09-10 11:37:47.835584', 'end_timestamp': '2026-09-10 11:38:00.607150', 'snapshot_counts': 5}
Messages:
 -> (role = system, content = You are an assistant that can help with:
, tool_calls = None)
 -> (role = user, content = Pick one random Pokemon!, tool_calls = None)
 -> (role = assistant, content = None, tool_calls = [ChatCompletionMessageFunctionToolCall(id='call_5hGjDCIWdFgqsMQ7zvaTrN3P', function=Function(arguments='{}', name='get_random_pokemon'), type='function')])
 -> (role = tool, content = "{'name': 'gastly', 'url': 'https://pokeapi.co/api/v2/pokemon/92/'}", tool_calls = None)
 -> (role = assistant, content = I picked a random Pokémon for you: **Gastly**! 

You can find more about Gastly [here](https://pokeapi.co/api/v2/pokemon/92/)., tool_calls = None)


In [ ]:
runs = agent.get_session_runs(session_id)
for run_object in runs:
    print(run_object)
    for snp in run_object.snapshots:
        print(f"-> {snp}")
    print("\n")

Run('48794dc0-deed-4104-9334-e1077cbb9a3a')
-> Snapshot('0899855d-0912-42e9-a6db-273318919d8a') @ [2026-09-10 11:37:47.835860]: __entry__.State({'user_query': 'Pick one random Pokemon!', 'instructions': 'You are an assistant that can help with:\n', 'messages': [], 'current_tool_calls': None, 'session_id': 'external_tools'})
-> Snapshot('32500ef6-de0e-49f1-b258-2ffe4e1c3e6c') @ [2026-09-10 11:37:47.836153]: message_prep.State({'user_query': 'Pick one random Pokemon!', 'instructions': 'You are an assistant that can help with:\n', 'messages': [SystemMessage(role='system', content='You are an assistant that can help with:\n'), UserMessage(role='user', content='Pick one random Pokemon!')], 'current_tool_calls': None, 'session_id': 'external_tools'})
-> Snapshot('ab2af71b-50c1-4328-8ec7-022622cd1c1b') @ [2026-09-10 11:37:57.807519]: llm_processor.State({'user_query': 'Pick one random Pokemon!', 'instructions': 'You are an assistant that can help with:\n', 'messages': [SystemMessage(role='sys